# Beginner Python Tutorial: Base Models, Reasoning, and Mixture of Experts

This notebook teaches the ideas using **small toy models** that can run on a normal computer.

It is designed to help you understand the architecture used in your project:

```text
Base GPT-OSS model
        ↓
Stage 1: domain-specific language adaptation
        ↓
Stage 2: A&P task fine-tuning
```

Important: the Python examples here are simplified demonstrations. They are **not** implementations of a large open-weight LM.

## Learning goals

By the end, you should understand:

1. What a **base language model** is.
2. How text becomes **tokens and numbers**.
3. How next-token prediction works.
4. What people mean by a **reasoning model**.
5. What a **Mixture-of-Experts (MoE)** model is.
6. What a **router** does.
7. How Stage 1 and Stage 2 change a base model.

# Part 1 — Text must become numbers

A language model cannot directly process words. A tokenizer converts text into tokens and token IDs.

Real tokenizers are complex. We will begin with a very small word-level tokenizer.

In [ ]:
text = "the patient has stable liver function"

tokens = text.lower().split()
vocabulary = sorted(set(tokens))
token_to_id = {token: i for i, token in enumerate(vocabulary)}
id_to_token = {i: token for token, i in token_to_id.items()}

token_ids = [token_to_id[token] for token in tokens]

print("Tokens:", tokens)
print("Vocabulary:", vocabulary)
print("Token IDs:", token_ids)
print("Decoded:", " ".join(id_to_token[i] for i in token_ids))

### What happened?

```text
Human text → tokens → token IDs
```

A real model's tokenizer may split one word into several pieces. For example, a medical term such as `tacrolimus` may be represented by one token or several subword tokens.

# Part 2 — What is a base language model?

A **base model** is a model that has already learned general language patterns before your project-specific training begins.

Its most basic job is:

> Given previous tokens, assign probabilities to possible next tokens.

We can demonstrate this with a tiny count-based language model.

In [ ]:
training_sentences = [
    "the patient has stable graft function",
    "the patient has normal liver enzymes",
    "the patient takes tacrolimus",
    "the graft function is stable",
    "tacrolimus level is stable",
]

from collections import defaultdict, Counter

next_word_counts = defaultdict(Counter)

for sentence in training_sentences:
    words = ["<START>"] + sentence.split() + ["<END>"]
    for current_word, next_word in zip(words, words[1:]):
        next_word_counts[current_word][next_word] += 1

def next_word_probabilities(current_word):
    counts = next_word_counts[current_word]
    total = sum(counts.values())
    return {
        word: count / total
        for word, count in counts.items()
    }

print("Possible words after 'the':")
print(next_word_probabilities("the"))

print("\nPossible words after 'patient':")
print(next_word_probabilities("patient"))

This toy model only remembers how often one word followed another.

A real LLM is much more powerful because it uses:

- many transformer layers;
- attention;
- billions of learned parameters;
- a long context, rather than only one previous word.

But the central objective remains similar: **predict the next token**.

In [ ]:
import random

def generate_toy_sentence(max_words=15):
    current = "<START>"
    output = []

    for _ in range(max_words):
        choices = next_word_counts[current]
        words = list(choices.keys())
        weights = list(choices.values())
        next_word = random.choices(words, weights=weights, k=1)[0]

        if next_word == "<END>":
            break

        output.append(next_word)
        current = next_word

    return " ".join(output)

for _ in range(5):
    print(generate_toy_sentence())

## Why is GPT-OSS called the base model in your project?

In your project, `example/base-model-id` already had general language and instruction-following abilities. Your project did not train language from zero.

You started with the base model and added specialty training:

```text
General base model
    + domain-specific adaptation
    + A&P task training
```

# Part 3 — Parameters and training

A parameter is a number inside a model.

Training changes parameters so that correct next tokens receive higher probability.

The following tiny example learns a relationship using gradient descent. It is not a language model, but it demonstrates the learning process.

In [ ]:
# We want a model to learn y = 2x + 1

x_values = [0, 1, 2, 3, 4]
y_values = [1, 3, 5, 7, 9]

weight = 0.0
bias = 0.0
learning_rate = 0.01

for epoch in range(1000):
    total_loss = 0.0
    grad_weight = 0.0
    grad_bias = 0.0

    for x, y_true in zip(x_values, y_values):
        y_pred = weight * x + bias
        error = y_pred - y_true
        total_loss += error ** 2

        grad_weight += 2 * error * x
        grad_bias += 2 * error

    n = len(x_values)
    weight -= learning_rate * grad_weight / n
    bias -= learning_rate * grad_bias / n

print("Learned weight:", round(weight, 3))
print("Learned bias:", round(bias, 3))
print("Prediction for x=5:", round(weight * 5 + bias, 3))

The same general cycle appears in LLM training:

```text
Input tokens
    ↓
Model prediction
    ↓
Compare with correct next token
    ↓
Calculate loss
    ↓
Calculate gradients
    ↓
Update trainable parameters
```

GPT-OSS has billions of parameters, whereas our toy example has only two.

# Part 4 — What is a reasoning model?

A reasoning model is still a language model. It still predicts tokens.

The difference is that it is trained or configured to spend more computation organizing a problem before producing the final response.

A useful beginner distinction is:

```text
Direct response:
input → answer

Reasoning-style response:
input → intermediate computation/planning → answer
```

This does **not** mean the model thinks exactly like a human or physician.

We will build a transparent toy example that separates a planner from a final answer generator.

In [ ]:
clinical_note = {
    "alt": 22,
    "ast": 20,
    "bilirubin": 0.8,
    "tacrolimus_level": 5.2,
    "creatinine": 85,
}

def direct_answer(note):
    return "The patient appears stable."

print(direct_answer(clinical_note))

The direct answer is short, but it does not show that the program checked each important area.

Now we add an explicit intermediate checklist.

In [ ]:
def make_clinical_checklist(note):
    checklist = {
        "graft_function_checked": all(
            key in note for key in ["alt", "ast", "bilirubin"]
        ),
        "immunosuppression_checked": "tacrolimus_level" in note,
        "kidney_function_checked": "creatinine" in note,
    }
    return checklist

def produce_final_summary(note, checklist):
    if not all(checklist.values()):
        return "Important information is missing."

    liver_stable = (
        note["alt"] < 40
        and note["ast"] < 40
        and note["bilirubin"] < 1.2
    )

    if liver_stable:
        return (
            "Graft markers in this toy example are stable. "
            "Tacrolimus and kidney function were also reviewed."
        )

    return "Some graft markers require review."

checklist = make_clinical_checklist(clinical_note)
final_summary = produce_final_summary(clinical_note, checklist)

print("Intermediate checklist:", checklist)
print("Final answer:", final_summary)

### What this teaches

The intermediate checklist is a **toy analogy** for extra computation before answering.

Real reasoning models do not necessarily use this exact checklist design. Their intermediate processing is learned and much more complex.

Also, an LLM's intermediate reasoning is not proof that the answer is correct. A model can produce plausible reasoning and still make mistakes.

# Part 5 — What is Mixture of Experts?

A Mixture-of-Experts model contains several internal neural-network blocks called **experts**.

For each token, a **router** decides which experts should process it.

```text
Token representation
        ↓
Router scores experts
        ↓
Select a small number of experts
        ↓
Experts transform the token representation
        ↓
Combine their outputs
```

The experts are not usually labelled clearly as “medicine,” “coding,” or “grammar.” They learn different numerical patterns during training.

In [ ]:
import numpy as np

# A token is represented here by a small vector.
token_vector = np.array([1.0, 0.5])

# Three toy experts. Each expert applies a different matrix.
expert_1 = np.array([[1.0, 0.0],
                     [0.0, 1.0]])

expert_2 = np.array([[0.5, 1.0],
                     [1.0, 0.5]])

expert_3 = np.array([[1.5, -0.5],
                     [0.5, 1.5]])

experts = [expert_1, expert_2, expert_3]

for i, expert in enumerate(experts, start=1):
    output = expert @ token_vector
    print(f"Expert {i} output:", output)

Each expert transforms the same token representation differently.

Now we need a router.

In [ ]:
# Router weights: each row produces a score for one expert.
router_weights = np.array([
    [1.0, 0.2],
    [0.1, 1.2],
    [0.7, 0.7],
])

router_scores = router_weights @ token_vector

def softmax(values):
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()

router_probabilities = softmax(router_scores)

print("Router scores:", router_scores)
print("Router probabilities:", router_probabilities)

The probabilities show how strongly the router prefers each expert for this token.

Many MoE systems select only the top one or top few experts. This is called **sparse routing**.

In [ ]:
top_k = 2
selected_indices = np.argsort(router_probabilities)[-top_k:]

print("Selected expert indices:", selected_indices)

combined_output = np.zeros_like(token_vector)

for index in selected_indices:
    expert_output = experts[index] @ token_vector
    weight = router_probabilities[index]
    combined_output += weight * expert_output

print("Combined selected-expert output:", combined_output)

## Why is MoE efficient?

Imagine a model has ten experts but uses only two for each token.

The model can have a large total capacity, while avoiding the cost of running every expert for every token.

In the uploaded project document, a large open-weight LM is described as having about 20.9 billion total parameters, with about 3.6 billion active per token because of its MoE design.

This does not mean the remaining parameters disappear. They are available, but the router does not activate all of them for that token.

# Part 6 — Watch the router change for different tokens

We will create two toy token vectors and observe how the router chooses experts differently.

In [ ]:
token_vectors = {
    "medical_like_token": np.array([1.5, 0.2]),
    "code_like_token": np.array([0.2, 1.5]),
    "balanced_token": np.array([0.8, 0.8]),
}

for name, vector in token_vectors.items():
    scores = router_weights @ vector
    probabilities = softmax(scores)
    selected = np.argsort(probabilities)[-2:]

    print(f"\n{name}")
    print(" vector:", vector)
    print(" probabilities:", np.round(probabilities, 3))
    print(" selected experts:", selected)

Important: the labels `medical_like_token` and `code_like_token` are only names we assigned to toy vectors.

In a real MoE model, the experts are learned automatically and may not have interpretable human labels.

# Part 7 — A complete toy MoE layer

Now we combine routing and expert computation into one Python function.

In [ ]:
def toy_moe_layer(token_vector, experts, router_weights, top_k=2):
    if top_k < 1 or top_k > len(experts):
        raise ValueError("top_k must be between 1 and the number of experts")

    scores = router_weights @ token_vector
    probabilities = softmax(scores)
    selected_indices = np.argsort(probabilities)[-top_k:]

    output = np.zeros(experts[0].shape[0], dtype=float)

    # Re-normalize only across selected experts.
    selected_weights = probabilities[selected_indices]
    selected_weights = selected_weights / selected_weights.sum()

    for index, weight in zip(selected_indices, selected_weights):
        output += weight * (experts[index] @ token_vector)

    return {
        "output": output,
        "router_probabilities": probabilities,
        "selected_experts": selected_indices,
    }

result = toy_moe_layer(
    token_vector=np.array([1.0, 0.5]),
    experts=experts,
    router_weights=router_weights,
    top_k=2,
)

print("Output:", result["output"])
print("Probabilities:", result["router_probabilities"])
print("Selected experts:", result["selected_experts"])

# Part 8 — How reasoning and MoE are different

These concepts answer different questions.

| Concept | Question it answers |
|---|---|
| Base model | What model do we start with? |
| Reasoning model | Does the model use additional computation before the final response? |
| Mixture of Experts | Which internal expert blocks process each token? |
| Router | How are experts selected? |
| Fine-tuning | How do we specialize the model for our task? |

A model can be:

- reasoning-focused without MoE;
- MoE without strong reasoning behavior;
- both reasoning-focused and MoE.

# Part 9 — Connecting this to your domain-adapted model

Your project can be understood as:

```text
a large open-weight LM base model
    - already knows general language
    - has a Mixture-of-Experts architecture
    - supports reasoning-style output
            ↓
Stage 1: DAPT
    - reads clinical notes
    - continues next-token prediction
    - learns domain vocabulary and note style
            ↓
Stage 2: SFT
    - input: visit note without A&P
    - target: clinician-written A&P
    - learns the specific drafting task
```

The uploaded document emphasizes that Stage 1 and Stage 2 teach different abilities: domain language versus A&P generation.

# Part 10 — A toy version of Stage 1 versus Stage 2

This is a conceptual simulation, not actual neural-network training.

In [ ]:
base_knowledge = {
    "patient": ["has", "is", "takes"],
    "liver": ["function", "disease"],
}

stage1_domain_updates = {
    "tacrolimus": ["level", "dose", "trough"],
    "graft": ["function", "stable", "rejection"],
    "myfortic": ["dose", "continue"],
}

stage2_task_template = {
    "Graft function": "Summarize liver enzymes and graft status.",
    "Immunosuppression": "Summarize medications, levels, and changes.",
    "Follow-up": "State monitoring and next review.",
}

print("Base model patterns:")
print(base_knowledge)

print("\nAfter Stage 1, domain-specific patterns are added:")
print(stage1_domain_updates)

print("\nAfter Stage 2, the model is trained to produce this task structure:")
for section, instruction in stage2_task_template.items():
    print(f"- {section}: {instruction}")

# Part 11 — Exercises

Try these exercises by editing the code.

### Exercise 1
Add more training sentences to the tiny next-word model. Does its generated text change?

### Exercise 2
Change the router weights. Which experts are selected?

### Exercise 3
Change `top_k` from 2 to 1 or 3. How does the combined output change?

### Exercise 4
Add a fourth expert matrix.

### Exercise 5
Modify the clinical checklist so it also checks whether cancer surveillance information is present.

# Final summary

## Base model
A pretrained model that already knows general language. It is the starting point before your domain-specific adaptation.

## Reasoning model
A language model that uses additional computation or intermediate processing before producing the final answer. It still predicts tokens and can still be wrong.

## Mixture of Experts
An architecture containing multiple expert neural-network blocks. A router selects only a subset for each token.

## Your project
You started with a large open-weight LM, then used Stage 1 to teach domain-specific note language and Stage 2 to teach note-to-A&P generation.